# 99_freeze — 사전 고정 기록 (담당: core, 금요일 1회)

Test를 열기 전에 실행합니다. git 대신 **파일 해시와 복사본**으로 "이 설정으로 고정했다"는 증거를 남깁니다.

1. `00_config`가 `STAGE = "final"`이고 미결 0개인지 확인합니다(아니면 멈춤).
2. 06_collect의 점검처럼 세 모형 결과가 같은 버전인지 확인합니다.
3. `freeze/freeze_<날짜시각>/`에 notebooks·contract·각 모형 summary를 복사하고, 모든 파일의 sha256과 config 전체를 `FREEZE_RECORD.json`에 씁니다.
4. 이 폴더를 Drive `lendingclub/freeze/`에 그대로 올리고, 해시 두 개를 팀에 공지합니다. 그 뒤에 `00b_test_prep`을 실행합니다.

`DRY_RUN = True`면 STAGE 확인을 건너뛰고 `freeze/_dryrun_…`에 시험 기록만 만듭니다.

In [ ]:
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
import json, hashlib, shutil, time

DRY_RUN = False
if not DRY_RUN:
    assert STAGE == "final" and not PENDING, f"STAGE='final'이고 미결 0개여야 합니다. 지금: STAGE={STAGE}, 미결={list(PENDING)}"
stamp = time.strftime("%Y%m%d_%H%M")
FREEZE = ROOT / "freeze" / (f"_dryrun_{stamp}" if DRY_RUN else f"freeze_{stamp}")
FREEZE.mkdir(parents=True, exist_ok=False)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
print("기록 폴더:", FREEZE.relative_to(ROOT))

In [ ]:
# 결과 버전 확인 (세 모형 모두 지금 해시와 같아야 함)
now = dict(config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH)
outs = {}
for m in ("logit", "xgb", "xgb_mono"):
    p = OUT_DIR / m / "summary.json"
    s = json.load(open(p)) if p.exists() else None
    ok = s is not None and s.get("config_hash") == now["config_hash"] and s.get("lib_prep_hash") == now["lib_prep_hash"]
    outs[m] = dict(exists=s is not None, same_version=ok, a_star=(s or {}).get("a_star"), B=(s or {}).get("B"))
    print(m, outs[m])
if not DRY_RUN:
    assert all(v["same_version"] for v in outs.values()), "세 모형 결과가 모두 지금 버전이어야 합니다(06_collect로 확인)"


In [ ]:
# 복사와 기록
files = {}
for sub, pats in (("notebooks", ["*.ipynb"]), ("contract", ["*.json"])):
    (FREEZE / sub).mkdir()
    for pat in pats:
        for p in sorted((ROOT / sub).glob(pat)):
            shutil.copy2(p, FREEZE / sub / p.name); files[f"{sub}/{p.name}"] = sha(p)
for m in outs:
    for name in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv"):
        p = OUT_DIR / m / name
        if p.exists():
            (FREEZE / "outputs" / m).mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, FREEZE / "outputs" / m / name); files[f"outputs/{m}/{name}"] = sha(p)
record = dict(stamp=stamp, dry_run=DRY_RUN, stage=STAGE, pending=list(PENDING), **now,
              config=config_dict(), outputs=outs, sha256=files, meta=run_meta())
(FREEZE / "FREEZE_RECORD.json").write_text(json.dumps(record, ensure_ascii=False, indent=1, default=str))
print(f"파일 {len(files)}개 기록 → {FREEZE.relative_to(ROOT)}/FREEZE_RECORD.json")
print("공지할 값: config", now["config_hash"], "| lib_prep", now["lib_prep_hash"])